In [1]:
%load_ext autoreload
%autoreload 2

In [26]:
import asyncio
import urllib.parse
from urllib.parse import quote, urlencode
import httpx
from bs4 import BeautifulSoup

In [16]:
class LinkedInJobParser:

    def __init__(self):
        self.base_url = "https://www.linkedin.com/jobs/search"
        self.headers = {
            "User-Agent": (
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0"
                " Safari/537.36"
            ),
            "Accept-Language": "en-US,en;q=0.9",
        }

    async def fetch_jobs(
        self, keywords: str, location: str = "", limit: int = 10, page_num: int = 1
    ) -> list[dict]:
        """Парсинг свежих вакансий с публичного API LinkedIn."""
        jobs = []

        

        params = {
            "keywords": keywords,
            "location": location,
            "geoId": "",
            "trk": "public_jobs_jobs-search-bar_search-submit",
            "position": 1,
            "pageNum": page_num,
            "f_TPR": "r86400",  # Дополнительно: вакансии за последние 24 часа
        }

        # urlencode кодирует пробелы и спецсимволы автоматически
        full_url = f"{self.base_url}?{urlencode(params)}"

        async with httpx.AsyncClient(headers=self.headers, timeout=10.0) as client:
            response = await client.get(full_url)

            if response.status_code != 200:
                print(f"Ошибка запроса: Status {response.status_code}")
                return jobs

            soup = BeautifulSoup(response.text, "html.parser")
            cards = soup.find_all("li")

            for card in cards[:limit]:
                title_elem = card.find("h3", class_="base-search-card__title")
                company_elem = card.find(
                    "h4", class_="base-search-card__subtitle"
                )
                location_elem = card.find(
                    "span", class_="job-search-card__location"
                )
                link_elem = card.find("a", class_="base-card__full-link")

                if not (title_elem and link_elem):
                    continue

                # Очистка ссылки от utm-меток
                raw_url = link_elem.get("href", "").split("?")[0]

                job_data = {
                    "title": title_elem.get_text(strip=True),
                    "company": (
                        company_elem.get_text(strip=True)
                        if company_elem
                        else "N/A"
                    ),
                    "location": (
                        location_elem.get_text(strip=True)
                        if location_elem
                        else "N/A"
                    ),
                    "url": raw_url,
                }
                jobs.append(job_data)

        print(f"Найдено {len(jobs)} вакансий по запросу '{keywords}'")
        return jobs

In [17]:
# Пример автономного запуска
async def main():
    parser = LinkedInJobParser()
    vacancies = await parser.fetch_jobs(
        keywords="Senior Machine Learning Engineer",
        limit=5,
    )

    for item in vacancies:
        print(f"Позиция: {item['title']}")
        print(f"Компания: {item['company']}")
        print(f"Локация: {item['location']}")
        print(f"Ссылка: {item['url']}\n---")

In [18]:
if __name__ == "__main__":
    await main()

Найдено 0 вакансий по запросу 'Senior Machine Learning Engineer'


In [30]:
base_url = "https://www.linkedin.com/jobs/search"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0"
        " Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

params = {
    "keywords": "Senior Machine Learning Engineer",
    "location": "Serbia",
    "geoId": "",
    "trk": "public_jobs_jobs-search-bar_search-submit",
    "position": 1,
    "pageNum": 0,
    "f_TPR": "r86400",  # Дополнительно: вакансии за последние 24 часа
}

# urlencode кодирует пробелы и спецсимволы автоматически
full_url = f"{base_url}?{urlencode(params, quote_via=quote)}"

async with httpx.AsyncClient(headers=headers, timeout=10.0) as client:
    response = await client.get(full_url)

    if response.status_code != 200:
        print(f"Ошибка запроса: Status {response.status_code}")

    soup = BeautifulSoup(response.text, "html.parser")

In [31]:
full_url

'https://www.linkedin.com/jobs/search?keywords=Senior%20Machine%20Learning%20Engineer&location=Serbia&geoId=&trk=public_jobs_jobs-search-bar_search-submit&position=1&pageNum=0&f_TPR=r86400'

In [ ]:
"https://www.linkedin.com/jobs/search?keywords=Senior%20Machine%20Learning%20Engineer&location=Serbia&geoId=&trk=public_jobs_jobs-search-bar_search-submit&position=1&pageNum=0"

In [32]:
soup

<!DOCTYPE html>

<html lang="en">
<head>
<meta content="d_jobs_guest_search" name="pageKey"/>
<meta content="max-image-preview:large, noarchive" name="robots"/>
<meta content="max-image-preview:large, archive" name="bingbot"/>
<!-- --> <meta content="urlType=jserp_custom;emptyResult=false" name="linkedin:pageTag"/>
<meta content="en_US" name="locale"/>
<!-- --> <meta data-app-version="2.0.3185" data-browser-id="0d410288-34b4-4bba-815d-ad3b4e41b2df" data-call-tree-id="AAZcKnLJswOB5gA1DN5fww==" data-dfp-member-lix-treatment="control" data-disable-jsbeacon-pagekey-suffix="false" data-dna-member-lix-treatment="enabled" data-enable-page-view-heartbeat-tracking="" data-human-member-lix-treatment="enabled" data-is-bot="false" data-is-epd-audit-event-enabled="false" data-is-feed-sponsored-tracking-kill-switch-enabled="false" data-member-id="0" data-multiproduct-name="jobs-guest-frontend" data-network-interceptor-lix-value="control" data-page-instance="urn:li:page:d_jobs_guest_search;lh873tnVSL